## Insurance Dataset Generator

Generates a synthetic **insurance** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `insurance` | `policyholders` | ~22K | `claims` | 100K-500K | Multi-policy (15K people), all 50 states + DC, pattern-based fraud, subrogation |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `insurance` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.insurance') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.insurance');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(37)
random.seed(37)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "insurance"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = date(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Policyholders table (~22K rows from 15K unique people, multi-policy) ---
# All 50 US states with approximate population weights and premium factors
states_config = {
    "California": (12, 1.25, "West"), "Texas": (9, 0.95, "South"), "Florida": (7, 1.15, "South"),
    "New York": (6, 1.35, "Northeast"), "Pennsylvania": (4, 1.00, "Northeast"),
    "Illinois": (4, 1.05, "Midwest"), "Ohio": (4, 0.90, "Midwest"),
    "Georgia": (3, 0.95, "South"), "North Carolina": (3, 0.92, "South"),
    "Michigan": (3, 1.10, "Midwest"), "New Jersey": (3, 1.20, "Northeast"),
    "Virginia": (3, 0.95, "South"), "Washington": (2, 1.05, "West"),
    "Arizona": (2, 0.90, "West"), "Massachusetts": (2, 1.18, "Northeast"),
    "Colorado": (2, 1.00, "West"), "Tennessee": (2, 0.88, "South"),
    "Maryland": (2, 1.08, "Northeast"), "Minnesota": (2, 0.95, "Midwest"),
    "Oregon": (2, 1.02, "West"), "Indiana": (2, 0.88, "Midwest"),
    "Missouri": (2, 0.90, "Midwest"), "Wisconsin": (2, 0.92, "Midwest"),
    "Connecticut": (1, 1.15, "Northeast"), "Alabama": (1, 0.85, "South"),
    "South Carolina": (1, 0.88, "South"), "Louisiana": (1, 0.98, "South"),
    "Kentucky": (1, 0.85, "South"), "Oklahoma": (1, 0.82, "South"),
    "Iowa": (1, 0.88, "Midwest"), "Utah": (1, 0.90, "West"),
    "Nevada": (1, 0.95, "West"), "Arkansas": (1, 0.80, "South"),
    "Kansas": (1, 0.85, "Midwest"), "Mississippi": (1, 0.82, "South"),
    "Nebraska": (1, 0.85, "Midwest"), "New Mexico": (1, 0.88, "West"),
    "Idaho": (1, 0.85, "West"), "West Virginia": (1, 0.82, "South"),
    "Hawaii": (1, 1.22, "West"), "New Hampshire": (1, 1.05, "Northeast"),
    "Maine": (1, 1.00, "Northeast"), "Montana": (1, 0.85, "West"),
    "Rhode Island": (1, 1.10, "Northeast"), "Delaware": (1, 1.05, "Northeast"),
    "South Dakota": (1, 0.82, "Midwest"), "North Dakota": (1, 0.80, "Midwest"),
    "Alaska": (1, 1.15, "West"), "Vermont": (1, 1.02, "Northeast"),
    "Wyoming": (1, 0.80, "West"), "District of Columbia": (1, 1.30, "Northeast"),
}
state_names = list(states_config.keys())
state_pop_weights = [v[0] for v in states_config.values()]
state_premium_factors = {k: v[1] for k, v in states_config.items()}
state_regions = {k: v[2] for k, v in states_config.items()}

region_agents = {
    "West":      [f"AGT-{1000 + i}" for i in range(15)],
    "South":     [f"AGT-{1015 + i}" for i in range(18)],
    "Northeast": [f"AGT-{1033 + i}" for i in range(15)],
    "Midwest":   [f"AGT-{1048 + i}" for i in range(14)],
}

policy_types = ["Auto", "Homeowners", "Life", "Health", "Renters"]
policy_statuses = ["Active", "Lapsed", "Cancelled", "Expired"]

policy_params = {
    "Auto":        (9.8,  0.5, 5000,   100000,  5.0, 0.4),
    "Homeowners":  (12.2, 0.6, 100000, 1500000, 5.5, 0.5),
    "Life":        (12.0, 0.8, 25000,  2000000, 4.0, 0.6),
    "Health":      (10.5, 0.4, 10000,  500000,  5.8, 0.4),
    "Renters":     (8.5,  0.4, 5000,   75000,   3.0, 0.3),
}
deductible_options = {
    "Auto": [250, 500, 1000, 2000], "Homeowners": [500, 1000, 2500, 5000],
    "Life": [0], "Health": [500, 1000, 2000, 3000, 5000, 7500], "Renters": [200, 500, 1000],
}
deductible_weights = {
    "Auto": [15, 40, 35, 10], "Homeowners": [10, 35, 35, 20],
    "Life": [100], "Health": [5, 15, 30, 25, 15, 10], "Renters": [25, 50, 25],
}

def policy_weights_for_age(age):
    if age < 25:     return [40, 5, 5, 25, 25]
    elif age < 35:   return [30, 15, 10, 25, 20]
    elif age < 50:   return [28, 28, 18, 18, 8]
    elif age < 65:   return [25, 30, 22, 18, 5]
    else:            return [20, 25, 15, 35, 5]

# Persons table: one row per unique person; policy rows expand from multi-policy assignment
NUM_PEOPLE = 15000
persons_rows = []
person_extras = {}  # person_id -> (num_prior, num_policies)
for i in range(1, NUM_PEOPLE + 1):
    age = int(clamp(random.gauss(45, 14), 18, 85))
    gender = random.choice(["Male", "Female"])
    state = random.choices(state_names, weights=state_pop_weights)[0]
    if age < 25: marital = random.choices(["Single", "Married", "Divorced", "Widowed"], weights=[75, 20, 4, 1])[0]
    elif age < 40: marital = random.choices(["Single", "Married", "Divorced", "Widowed"], weights=[30, 50, 18, 2])[0]
    elif age < 60: marital = random.choices(["Single", "Married", "Divorced", "Widowed"], weights=[12, 55, 25, 8])[0]
    else: marital = random.choices(["Single", "Married", "Divorced", "Widowed"], weights=[8, 40, 22, 30])[0]
    credit = int(clamp(random.gauss(710, 75), 300, 850))
    num_prior = min(10, int(random.expovariate(0.8)))
    # How many policies: 60% have 1, 30% have 2, 10% have 3
    num_policies = random.choices([1, 2, 3], weights=[60, 30, 10])[0]
    person_name = fake.name()
    email = fake.email()
    persons_rows.append(Row(
        person_id=i,
        person_name=person_name,
        email=email,
        age=age,
        gender=gender,
        marital_status=marital,
        state=state,
        credit_score=credit,
    ))
    person_extras[i] = (num_prior, num_policies)

persons_df = spark.createDataFrame(persons_rows)
persons_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.persons")
print(f"✔ Created {CATALOG_SCHEMA}.persons ({persons_df.count()} rows)")

person_by_id = {r.person_id: r for r in persons_rows}

policy_id_counter = 0
policyholders = []
for person_id in range(1, NUM_PEOPLE + 1):
    pr = person_by_id[person_id]
    age, gender, state, marital, credit = pr.age, pr.gender, pr.state, pr.marital_status, pr.credit_score
    name, email = pr.person_name, pr.email
    num_prior, num_policies = person_extras[person_id]
    assigned_types = []
    for p_idx in range(num_policies):
        # Avoid duplicate policy types for same person
        remaining_types = [t for t in policy_types if t not in assigned_types]
        if not remaining_types:
            break
        weights = policy_weights_for_age(age)
        adj_weights = [weights[policy_types.index(t)] if t in remaining_types else 0 for t in policy_types]
        if sum(adj_weights) == 0:
            break
        policy_type = random.choices(policy_types, weights=adj_weights)[0]
        assigned_types.append(policy_type)
        policy_id_counter += 1

        c_mu, c_sig, c_lo, c_hi, p_mu, p_sig = policy_params[policy_type]
        coverage = float(round(clamp(random.lognormvariate(c_mu, c_sig), c_lo, c_hi), 2))

        age_factor = 1.0 + (age - 30) * 0.005 if age > 30 else 1.0
        if age < 25 and policy_type == "Auto": age_factor = 1.35
        state_factor = state_premium_factors.get(state, 1.0)
        marital_factor = 0.92 if marital == "Married" else 1.0
        premium = float(round(clamp(random.lognormvariate(p_mu, p_sig) * age_factor * state_factor * marital_factor, 10, 3000), 2))
        if num_policies > 1:
            premium = float(round(premium * 0.90, 2))

        deductible = random.choices(deductible_options[policy_type], weights=deductible_weights[policy_type])[0]

        base_risk = clamp(random.betavariate(2.5, 4.0) * 100, 1, 100)
        credit_adj = (750 - credit) * 0.03
        claims_adj = num_prior * 5
        age_risk_adj = abs(age - 45) * 0.15
        risk = round(float(clamp(base_risk + credit_adj + claims_adj + age_risk_adj, 1, 100)), 1)

        start_date = date(2019, 1, 1) + timedelta(days=random.randint(0, 2500))
        tenure = (NOW - start_date).days / 365.25
        if tenure > 4: p_status = random.choices(policy_statuses, weights=[82, 5, 6, 7])[0]
        elif tenure > 2: p_status = random.choices(policy_statuses, weights=[75, 10, 10, 5])[0]
        else: p_status = random.choices(policy_statuses, weights=[65, 15, 15, 5])[0]

        has_bundle = num_policies > 1
        agent_region = state_regions.get(state, "West")
        agent = random.choice(region_agents[agent_region])

        policyholders.append(Row(
            policy_id=policy_id_counter,
            policyholder_id=person_id,
            policyholder_name=name,
            email=email,
            age=age,
            gender=gender,
            marital_status=marital,
            state=state,
            credit_score=credit,
            policy_type=policy_type,
            policy_status=p_status,
            coverage_amount=coverage,
            monthly_premium=premium,
            deductible=float(deductible),
            risk_score=risk,
            num_prior_claims=num_prior,
            policy_start_date=start_date,
            has_bundled_discount=has_bundle,
            assigned_agent=agent
        ))

ph_lookup = {p.policy_id: p for p in policyholders}
person_policies = {}
for p in policyholders:
    person_policies.setdefault(p.policyholder_id, []).append(p)

policyholders_df = spark.createDataFrame(policyholders)
policyholders_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.policyholders")
print(f"✔ Created {CATALOG_SCHEMA}.policyholders ({policyholders_df.count()} rows, {NUM_PEOPLE} unique people)")

# --- Claims table (randomized ~100K-500K rows) ---
claim_types_by_policy = {
    "Auto":       ["Collision", "Comprehensive", "Liability", "Uninsured Motorist", "Theft", "Windshield"],
    "Homeowners": ["Water Damage", "Fire", "Theft", "Wind/Hail", "Liability", "Structural"],
    "Life":       ["Death Benefit", "Accidental Death", "Terminal Illness"],
    "Health":     ["Inpatient", "Outpatient", "Prescription", "Emergency", "Specialist", "Lab/Diagnostic"],
    "Renters":    ["Theft", "Water Damage", "Fire", "Liability", "Vandalism"],
}
claim_type_weights = {
    "Auto":       [35, 15, 20, 5, 8, 17],
    "Homeowners": [30, 10, 12, 25, 8, 15],
    "Life":       [70, 20, 10],
    "Health":     [15, 30, 25, 10, 12, 8],
    "Renters":    [30, 25, 10, 15, 20],
}
claim_statuses = ["Approved", "Denied", "Pending", "Under Investigation", "Settled", "Closed"]
claim_amount_params = {
    "Auto":       (7.0, 1.0, 200, 75000),
    "Homeowners": (8.5, 1.2, 500, 500000),
    "Life":       (11.5, 0.8, 10000, 2000000),
    "Health":     (7.2, 1.3, 50, 200000),
    "Renters":    (5.8, 0.9, 100, 30000),
}

claim_pool = []
for p in policyholders:
    if p.policy_status != "Active":
        claim_pool.append(p.policy_id)
    else:
        weight = max(1, int(p.risk_score / 20))
        claim_pool.extend([p.policy_id] * weight)

# Track claims per policy for fraud velocity detection
policy_claim_count = {p.policy_id: 0 for p in policyholders}

claims = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    claim_ts = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 730),
                 hours=random.randint(0, 23), minutes=random.randint(0, 59))
    pol_id = random.choice(claim_pool)
    ph = ph_lookup[pol_id]
    pol_type = ph.policy_type
    policy_claim_count[pol_id] += 1

    ct_options = claim_types_by_policy[pol_type]
    ct_weights = claim_type_weights.get(pol_type, None)
    claim_type = random.choices(ct_options, weights=ct_weights)[0] if ct_weights else random.choice(ct_options)

    mu, sigma, lo, hi = claim_amount_params[pol_type]
    claim_amt = float(round(clamp(random.lognormvariate(mu, sigma), lo, hi), 2))

    if claim_amt > 20000 or ph.risk_score > 70:
        status = random.choices(claim_statuses, weights=[25, 15, 15, 15, 22, 8])[0]
    else:
        status = random.choices(claim_statuses, weights=[38, 10, 14, 6, 27, 5])[0]

    if status in ("Approved", "Settled", "Closed"):
        approval_ratio = clamp(random.betavariate(6.0, 2.5), 0.3, 1.0)
        approved_amt = float(round(claim_amt * approval_ratio, 2))
    elif status == "Denied":
        approved_amt = 0.0
    else:
        approved_amt = None

    days_resolve = int(clamp(random.lognormvariate(2.5, 0.9), 1, 365)) if status not in ("Pending", "Under Investigation") else None

    # Pattern-based fraud detection
    fraud_prob = 0.01
    days_since_inception = (claim_ts.date() - ph.policy_start_date).days
    if days_since_inception < 90: fraud_prob += 0.05  # early claim
    if claim_amt / ph.coverage_amount > 0.7: fraud_prob += 0.04  # near coverage limit
    if policy_claim_count[pol_id] > 3: fraud_prob += 0.03  # high frequency
    if claim_amt > 50000: fraud_prob += 0.03  # large amount
    fraud_suspected = random.random() < fraud_prob

    # Recovery/subrogation amount for settled claims
    recovery = 0.0
    if status in ("Approved", "Settled", "Closed") and approved_amt and claim_type in ("Collision", "Liability", "Theft", "Vandalism"):
        if random.random() < 0.25:
            recovery = float(round(approved_amt * clamp(random.betavariate(2, 5), 0.05, 0.60), 2))

    adj_region = state_regions.get(ph.state, "West")
    adjuster = f"ADJ-{2000 + hash(adj_region) % 12 + random.randint(0, 11)}"

    claims.append(Row(
        claim_id=50000 + i,
        policy_id=pol_id,
        policyholder_id=ph.policyholder_id,
        claim_date=claim_ts,
        policy_type=pol_type,
        claim_type=claim_type,
        claim_amount=claim_amt,
        approved_amount=approved_amt,
        recovery_amount=recovery,
        claim_status=status,
        days_to_resolve=days_resolve,
        adjuster_id=adjuster,
        fraud_suspected=fraud_suspected,
        police_report_filed=random.choices([True, False],
            weights=[60, 40] if claim_type in ("Theft", "Collision", "Vandalism", "Fire") else [10, 90])[0],
        description=f"{claim_type} claim - ref {random.randint(100000, 999999)}"
    ))

claims_df = spark.createDataFrame(claims)
claims_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.claims")
print(f"✔ Created {CATALOG_SCHEMA}.claims ({claims_df.count()} rows)")

payment_type_choices = ["Indemnity", "Loss Adjustment Expense", "Recovery"]
payment_methods = ["Check", "ACH", "Wire"]
payment_statuses = ["Processed", "Pending", "Returned"]

claim_payments = []
payment_id_counter = 0
for c in claims:
    approved = c.approved_amount
    if approved is None or approved <= 0:
        continue
    n_pay = random.randint(1, 3)
    raw_weights = [random.random() for _ in range(n_pay)]
    wsum = sum(raw_weights)
    amounts = [float(round(approved * w / wsum, 2)) for w in raw_weights]
    drift = round(approved - sum(amounts), 2)
    amounts[-1] = float(round(amounts[-1] + drift, 2))
    claim_ts = c.claim_date
    for k in range(n_pay):
        payment_id_counter += 1
        pay_dt = claim_ts + timedelta(
            days=random.randint(1, 90),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
        )
        claim_payments.append(Row(
            payment_id=payment_id_counter,
            claim_id=c.claim_id,
            payment_date=pay_dt,
            payment_type=random.choice(payment_type_choices),
            amount=amounts[k],
            payment_method=random.choice(payment_methods),
            status=random.choice(payment_statuses),
        ))

claim_payments_df = spark.createDataFrame(claim_payments)
claim_payments_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.claim_payments")
print(f"✔ Created {CATALOG_SCHEMA}.claim_payments ({claim_payments_df.count()} rows)")

print("\n--- Persons (sample) ---")
display(persons_df.limit(5))
print("\n--- Policyholders (sample) ---")
display(policyholders_df.limit(5))
print("\n--- Claims (sample) ---")
display(claims_df.limit(5))
print("\n--- Claim payments (sample) ---")
display(claim_payments_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.insurance.persons", {
    "person_id":       "Unique person identifier (1..NUM_PEOPLE). Primary key",
    "person_name":     "Full name (generated via Faker)",
    "email":           "Email address (generated via Faker)",
    "age":             "Current age in years",
    "gender":          "Gender: Male or Female",
    "marital_status":  "Marital status: Single, Married, Divorced, or Widowed",
    "state":           "US state (all 50 states + DC, population-weighted)",
    "credit_score":    "Credit score (normal, mean 710, range 300-850)",
})

apply_comments(f"{CATALOG}.insurance.policyholders", {
    "policy_id":           "Unique identifier for the policy. One person can have multiple policies",
    "policyholder_id":     "Foreign key to persons.person_id. ~30% have 2 policies, ~10% have 3 (bundle analysis)",
    "policyholder_name":   "Full name (denormalized from persons for backward compatibility)",
    "email":               "Email (denormalized from persons)",
    "age":                 "Age in years (denormalized from persons)",
    "gender":              "Gender (denormalized from persons)",
    "marital_status":      "Marital status (denormalized from persons)",
    "state":               "US state (denormalized from persons)",
    "credit_score":        "Credit score (denormalized from persons)",
    "policy_type":         "Insurance line: Auto, Homeowners, Life, Health, or Renters",
    "policy_status":       "Policy status: Active, Lapsed, Cancelled, or Expired",
    "coverage_amount":     "Maximum coverage amount in USD",
    "monthly_premium":     "Monthly premium with age, state, and marital-status risk loading; 10% discount when multiple policies per person",
    "deductible":          "Deductible amount in USD",
    "risk_score":          "Underwriting risk score 1-100",
    "num_prior_claims":    "Number of prior claims filed",
    "policy_start_date":   "Policy effective start date (DateType)",
    "has_bundled_discount":"True if the person holds multiple policies; when True, monthly_premium includes a 10% bundled discount",
    "assigned_agent":      "Agent ID. Region-correlated (AGT-XXXX format)",
})

apply_comments(f"{CATALOG}.insurance.claims", {
    "claim_id":            "Unique identifier for the claim",
    "policy_id":           "Foreign key referencing policyholders.policy_id",
    "policyholder_id":     "Person-level foreign key for cross-policy analysis",
    "claim_date":          "Timestamp when the claim was filed (TimestampType)",
    "policy_type":         "Insurance line of the associated policy",
    "claim_type":          "Specific claim category",
    "claim_amount":        "Total claimed amount in USD",
    "approved_amount":     "Amount approved. NULL if pending, 0 if denied",
    "recovery_amount":     "Subrogation/recovery amount for applicable settled claims (Collision, Liability, Theft, Vandalism)",
    "claim_status":        "Status: Approved, Denied, Pending, Under Investigation, Settled, or Closed",
    "days_to_resolve":     "Business days from filing to resolution. NULL if still open",
    "adjuster_id":         "Claims adjuster ID (ADJ-XXXX format)",
    "fraud_suspected":     "Pattern-based fraud flag. Triggered by: early claims (<90 days), near-limit amounts, high frequency, large claims",
    "police_report_filed": "Whether a police report was filed",
    "description":         "Free-text claim description with reference number",
})

apply_comments(f"{CATALOG}.insurance.claim_payments", {
    "payment_id":     "Unique payment event identifier (sequential)",
    "claim_id":       "Foreign key to claims.claim_id",
    "payment_date":   "Timestamp of the payment (after claim_date)",
    "payment_type":   "Indemnity, Loss Adjustment Expense, or Recovery",
    "amount":         "Payment amount in USD; split across 1-3 rows sums to approved_amount",
    "payment_method": "Check, ACH, or Wire",
    "status":         "Processed, Pending, or Returned",
})

print(f"\n\u2705 All column comments applied for insurance schema")

spark.sql(f"ALTER TABLE {CATALOG}.insurance.persons ALTER COLUMN person_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.persons ADD CONSTRAINT pk_persons PRIMARY KEY (person_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.policyholders ALTER COLUMN policy_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.policyholders ADD CONSTRAINT pk_policyholders PRIMARY KEY (policy_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.policyholders ADD CONSTRAINT fk_policyholders_person_id FOREIGN KEY (policyholder_id) REFERENCES {CATALOG}.insurance.persons(person_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claims ALTER COLUMN claim_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claims ADD CONSTRAINT pk_claims PRIMARY KEY (claim_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claims ADD CONSTRAINT fk_claims_policy_id FOREIGN KEY (policy_id) REFERENCES {CATALOG}.insurance.policyholders(policy_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claim_payments ALTER COLUMN payment_id SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claim_payments ADD CONSTRAINT pk_claim_payments PRIMARY KEY (payment_id)")
spark.sql(f"ALTER TABLE {CATALOG}.insurance.claim_payments ADD CONSTRAINT fk_claim_payments_claim_id FOREIGN KEY (claim_id) REFERENCES {CATALOG}.insurance.claims(claim_id)")
print(f"\u2714 PK/FK constraints applied for insurance schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.insurance') IS
'Insurance sample dataset with realistic statistical distributions and Faker-generated PII. Entity tables: `persons` (15K rows, PK: person_id), `policyholders` (~22K rows, PK: policy_id, FK: policyholder_id → persons). Event tables: `claims` (100K-500K rows, PK: claim_id, FK: policy_id → policyholders), `claim_payments` (1-3 payments per approved claim, PK: payment_id, FK: claim_id → claims). Key features: Multi-policy bundled premium discount, all 50 states + DC, pattern-based fraud, subrogation.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`insurance` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to insurance schema ({remove_after_value})")